# 地方競馬 予想システム 〜 学習用（週1回などデータ更新時に実行）

このノートブックは**学習して、予想に必要なものを保存するだけ**。予想は「予想用（すぐ起動）」ノートブックで行う。

**手順**
1. Input に「元のKaggleデータセット」と「nar-2026-scrape（最新版）」と、あれば穴埋め収集の結果（nar-fill）を追加（Internet は不要）
2. 右上「Save Version」→「**Save & Run All (Commit)**」で実行（20分前後。ブラウザを閉じてもよい）
3. 完了すると Output に `keiba_bundle` フォルダ（モデル・過去走データ）ができる
4. 予想用ノートブックの Input に、このノートブックの Output を追加する（Add Input →「Your Work」→ このノートブック）

nar-2026-scrape を更新したら、このノートブックを Commit し直すだけで、予想用は次回起動時に最新のモデルを読み込む。

**2026/11 改良**: 評価は「〜前年で学習・最終年で採点」のまま、保存するモデルは最終年も含めた全データで学習し直す（最新の傾向を取り込むため。学習時間は1.5倍ほど）。⑤以降の最終年の成績は、保存するモデルではなく評価用モデルの数字。

## ⓪ 設定

In [ ]:
import os, glob, warnings
import numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

DATA_ROOT   = "/kaggle/input"
TEST_YEARS  = list(range(2016, 2026))  # 1年ずつ評価する年（各年とも「前年まで」のデータだけで学習）
BASE_END    = "2015-01-01"   # スピード指数の基準タイムはこの日より前のデータだけで作る（全ての評価年より過去）
USE_ODDS    = True          # True: オッズも特徴量に入れ、「市場の評価のどこがズレているか」を直接学習させる

## ① 読込（オッズ "2,012.0" のカンマで割れた行を直して読む）

In [ ]:
DIR = os.path.dirname(glob.glob(f"{DATA_ROOT}/**/keiba_results.csv", recursive=True)[0])

def fix_row(row):
    if len(row) == 19:
        return row[:15] + [row[15] + row[16]] + row[17:]
    return None

res = pd.read_csv(f"{DIR}/keiba_results.csv", engine="python", on_bad_lines=fix_row,
                  dtype={"race_id": str, "horse_id": str, "jockey_id": str})
races = pd.read_csv(f"{DIR}/keiba_races.csv", dtype={"race_id": str})
pays  = pd.read_csv(f"{DIR}/keiba_payouts.csv", dtype={"race_id": str, "horse_num": str})
# 追加データ（2026年収集分・過去の穴埋め分）。収集ノートブックの出力を Private データセットにして Add Input しておくと自動で結合
for tag in ["2026", "fill"]:
    add = {n: sorted(glob.glob(f"{DATA_ROOT}/**/keiba_{n}_{tag}.csv", recursive=True)) for n in ["results", "races", "payouts"]}
    if not all(add.values()):
        continue
    res = pd.concat([res] + [pd.read_csv(p, dtype={"race_id": str, "horse_id": str, "jockey_id": str}) for p in add["results"]],
                    ignore_index=True).drop_duplicates(["race_id", "number"], keep="last")
    races = pd.concat([races] + [pd.read_csv(p, dtype={"race_id": str}) for p in add["races"]],
                      ignore_index=True).drop_duplicates("race_id", keep="last")
    pays = pd.concat([pays] + [pd.read_csv(p, dtype={"race_id": str, "horse_num": str}) for p in add["payouts"]],
                     ignore_index=True).drop_duplicates(keep="last")
    print(f"追加データ（{tag}）を結合しました:", add["races"])
print("results:", res.shape, "/ races:", races.shape, "/ payouts:", pays.shape)

## ② 地方に絞り、race_id から日付を作る

地方の race_id は「年(4) + 場(2) + 月(2) + 日(2) + R(2)」。帯広（ばんえい, 場コード65）は別競技なので除外。
下の確認で「月」が 1〜12 だけ、「日」が 1〜31 だけになっていれば解釈は正しい。

In [ ]:
EXCLUDE_VENUES = {"65"}

def is_nar(rid):
    code = rid.str[4:6]
    return ~code.astype(int).between(1, 10) & ~code.isin(EXCLUDE_VENUES)

res   = res[is_nar(res["race_id"])].copy()
races = races[is_nar(races["race_id"])].copy()

mm = races["race_id"].str[6:8].astype(int)
dd = races["race_id"].str[8:10].astype(int)
print("月の範囲:", mm.min(), "〜", mm.max(), "/ 日の範囲:", dd.min(), "〜", dd.max())
print("月ごとのレース数:", mm.value_counts().sort_index().to_dict())
assert mm.between(1, 12).all() and dd.between(1, 31).all(), "月日として解釈できない値があります"

races["date"] = pd.to_datetime(races["race_id"].str[:4] + races["race_id"].str[6:10],
                               format="%Y%m%d", errors="coerce")
print("日付が作れなかったレース:", races["date"].isna().sum())
races = races.dropna(subset=["date"])
pays  = pays[pays["race_id"].isin(set(races["race_id"]))].copy()
print("地方レース数:", len(races), "/ 期間:", races["date"].min().date(), "〜", races["date"].max().date())

## ③ 整形

In [ ]:
race_cols = ["race_id", "date", "venue", "race_number", "course_type", "distance",
             "turn", "weather", "track_condition"]
df = res.merge(races[race_cols], on="race_id", how="inner")

# 着順: 取消・除外は出走していないので除く / 中止・失格は最下位扱い / "2(降)" は数字部分
df = df[~df["rank"].astype(str).isin(["取", "除"])].copy()
df["n_runners"] = df.groupby("race_id")["race_id"].transform("size")
df["finish"] = pd.to_numeric(df["rank"].astype(str).str.extract(r"^(\d+)")[0], errors="coerce")
df["finish"] = df["finish"].fillna(df["n_runners"]).astype(int)
df["win"]  = (df["finish"] == 1).astype(int)
df["top3"] = (df["finish"] <= 3).astype(int)
df["finish_ratio"] = (df["finish"] - 1) / (df["n_runners"] - 1).clip(lower=1)

# 数値化
for c in ["frame", "number", "weight", "distance", "race_number", "odds", "popularity", "last_3f"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["sex"] = df["sex_age"].astype(str).str[0]
df["age"] = pd.to_numeric(df["sex_age"].astype(str).str[1:], errors="coerce")
bw = df["horse_weight"].astype(str).str.extract(r"(\d+)\(([-+]?\d+)\)")
df["body_weight"] = pd.to_numeric(bw[0], errors="coerce")
df["bw_diff"]     = pd.to_numeric(bw[1], errors="coerce")
t = df["time"].astype(str).str.extract(r"(?:(\d+):)?(\d+\.\d)")
df["time_sec"] = pd.to_numeric(t[0], errors="coerce").fillna(0) * 60 + pd.to_numeric(t[1], errors="coerce")
pos = df["passing"].astype(str).str.split("-")
df["first_corner"] = pd.to_numeric(pos.str[0], errors="coerce")
df["horse_key"]  = df["horse_id"].fillna(df["horse_name"])
df["jockey_key"] = df["jockey_id"].fillna(df["jockey_name"])
df["month"] = df["date"].dt.month

# レース後にしか分からない「そのレースでの内容」（前走特徴量の材料。そのレースの特徴量には使わない）
df["time_behind"]  = df["time_sec"] - df.groupby("race_id")["time_sec"].transform("min")
df["last3f_rank"]  = df.groupby("race_id")["last_3f"].rank(pct=True)
df["corner1_ratio"] = df["first_corner"] / df["n_runners"]
print(df.shape)

## ④ 過去走の特徴量（必ず shift(1) で当該レースを除外）

スピード指数は「直近3年の基準タイム」と「その日の馬場差」で計算（2026/10 改良）。

F4（メンバーの中での位置: 5走平均・最高指数のメンバー内の偏差、最上位との差、メンバー上位3頭の平均との差）も追加（2026/11）。

In [ ]:
# --- スピード指数: 競馬場・コース・距離・馬場ごとの基準タイムとの差（50が平均、高いほど速い）---
# 基準タイムは BASE_END より前の5着以内の馬だけで作る（どの評価年から見ても過去のデータ）
BASE_KEYS = ["venue", "course_type", "distance", "track_condition"]
base = (df[(df["date"] < BASE_END) & (df["finish"] <= 5)]
        .groupby(BASE_KEYS)["time_sec"].agg(base_mean="mean", base_std="std", base_n="size").reset_index())
base = base[base["base_n"] >= 30]
df = df.drop(columns=["base_mean", "base_std", "base_n", "speed"], errors="ignore").merge(base, on=BASE_KEYS, how="left")
df["speed"] = (df["base_mean"] - df["time_sec"]) / df["base_std"] * 10 + 50
print("スピード指数が計算できた割合:", round(df["speed"].notna().mean(), 3))

# --- スピード指数の作り直し（2026/10 改良：比較実験で5回全勝）---
# 1) 基準タイムを「各年について、その前の3年」で作り直す（馬場の改修や新しい距離設定に追随。未来の年は使わない）
# 2) その日・その競馬場・コースの時計の速さ（馬場差）で補正する（レース後に分かる値で、次走以降の特徴量にだけ使う）
df["year"] = df["date"].dt.year
df["_bk"] = (df["venue"].astype(str) + "|" + df["course_type"].astype(str) + "|" + df["distance"].astype(str)
             + "|" + df["track_condition"].astype(str))
parts = []
for Y in sorted(df["year"].unique()):
    s_ = df[(df["year"] >= Y - 3) & (df["year"] <= Y - 1) & (df["finish"] <= 5)]
    st = s_.groupby("_bk")["time_sec"].agg(b2_mean="mean", b2_std="std", b2_n="size").reset_index()
    st = st[st["b2_n"] >= 30]
    st["year"] = Y
    parts.append(st)
df = df.drop(columns=["b2_mean", "b2_std", "b2_n"], errors="ignore").merge(pd.concat(parts, ignore_index=True), on=["_bk", "year"], how="left")
raw2 = ((df["b2_mean"] - df["time_sec"]) / df["b2_std"] * 10 + 50).fillna(df["speed"])
race_dev = raw2.where(df["finish"] <= 5).groupby(df["race_id"]).transform("mean") - 50
day = df["date"].astype(str) + "_" + df["venue"].astype(str) + "_" + df["course_type"].astype(str)
first = ~df["race_id"].duplicated()
day_dev = race_dev[first].groupby(day[first]).mean()
n_day = first.groupby(day).sum()
variant = day.map(day_dev).where(day.map(n_day) >= 3)
df["speed_old"] = df["speed"]
df["speed"] = raw2 - variant.fillna(0)
print("新しいスピード指数: 直近3年の基準が使えた割合", round(df["b2_mean"].notna().mean(), 3),
      "/ 馬場差で補正できた割合", round(variant.notna().mean(), 3), "/ 計算できた割合", round(df["speed"].notna().mean(), 3))

# 予想用ノートブックで結果を手入力したときに使う基準タイム（直近3年。馬場差の補正は無し）
LY = int(df["year"].max())
base = (df[(df["year"] >= LY - 2) & (df["finish"] <= 5)]
        .groupby(BASE_KEYS, observed=True)["time_sec"].agg(base_mean="mean", base_std="std", base_n="size").reset_index())
base = base[base["base_n"] >= 30]

df = df.sort_values(["date", "race_id", "number"]).reset_index(drop=True)
g = df.groupby("horse_key", sort=False)

def past_mean(col, n):
    s = g[col].shift(1)
    return s.groupby(df["horse_key"]).rolling(n, min_periods=1).mean().reset_index(level=0, drop=True)

for k in (1, 2, 3):
    df[f"prev_finish_{k}"] = g["finish"].shift(k)
df["prev_finish_ratio"]   = g["finish_ratio"].shift(1)
df["prev_speed"]          = g["speed"].shift(1)
df["avg_speed_5"]         = past_mean("speed", 5)
df["max_speed_5"]         = (g["speed"].shift(1).groupby(df["horse_key"]).rolling(5, min_periods=1).max()
                             .reset_index(level=0, drop=True))
df["avg_finish_ratio_5"]  = past_mean("finish_ratio", 5)
df["top3_rate_5"]         = past_mean("top3", 5)
df["win_rate_10"]         = past_mean("win", 10)
df["prev_time_behind"]    = g["time_behind"].shift(1)
df["avg_time_behind_5"]   = past_mean("time_behind", 5)
df["prev_last3f_rank"]    = g["last3f_rank"].shift(1)
df["avg_last3f_rank_5"]   = past_mean("last3f_rank", 5)
df["prev_corner1_ratio"]  = g["corner1_ratio"].shift(1)
df["avg_corner1_ratio_5"] = past_mean("corner1_ratio", 5)
df["prev_popularity"]     = g["popularity"].shift(1)
df["prev_log_odds"]       = np.log1p(g["odds"].shift(1))
df["n_races"]             = g.cumcount()
df["days_since_last"]     = (df["date"] - g["date"].shift(1)).dt.days
df["dist_diff"]           = df["distance"] - g["distance"].shift(1)
df["weight_diff"]         = df["weight"] - g["weight"].shift(1)
df["same_venue_as_prev"]  = (df["venue"] == g["venue"].shift(1)).astype(float)

# 騎手・調教師の通算成績（前日までの成績のみ。平滑化つき）
K = 30
def past_rate(key, target):
    daily = (df.groupby([key, "date"])[target].agg(["sum", "count"])
               .reset_index().sort_values([key, "date"]))
    gg = daily.groupby(key)
    past_sum = gg["sum"].cumsum() - daily["sum"]
    past_cnt = gg["count"].cumsum() - daily["count"]
    p = df[target].mean()
    daily[f"{key}_{target}_rate"] = (past_sum + p * K) / (past_cnt + K)
    return daily[[key, "date", f"{key}_{target}_rate"]]

for key, target in [("jockey_key", "top3"), ("jockey_key", "win"), ("trainer", "top3")]:
    df = df.merge(past_rate(key, target), on=[key, "date"], how="left")

# レース内での相対位置（メンバーの中で強いか）
for c in ["avg_finish_ratio_5", "top3_rate_5", "avg_time_behind_5", "jockey_key_top3_rate",
          "avg_speed_5", "max_speed_5"]:
    df[f"{c}_inrace"] = df.groupby("race_id")[c].rank(pct=True)

# F4 メンバーの中での位置（2026/11 改良：3年×乱数10回の検証で26勝4敗）
for c in ["avg_speed_5", "max_speed_5"]:
    gc = df.groupby("race_id")[c]
    df[f"{c}_z"] = (df[c] - gc.transform("mean")) / gc.transform("std").replace(0, np.nan)
df["max_speed_5_gap"] = df["max_speed_5"] - df.groupby("race_id")["max_speed_5"].transform("max")
rk = df.groupby("race_id")["max_speed_5"].rank(ascending=False, method="first")
df["field_level"] = df["max_speed_5"].where(rk <= 3).groupby(df["race_id"]).transform("mean")
df["max_speed_5_vs_field"] = df["max_speed_5"] - df["field_level"]
F4_COLS = ["avg_speed_5_z", "max_speed_5_z", "max_speed_5_gap", "field_level", "max_speed_5_vs_field"]

cat_cols = ["venue", "course_type", "turn", "weather", "track_condition", "sex"]
for c in cat_cols:
    df[c] = df[c].astype("category")

features = (
    ["frame", "number", "weight", "age", "body_weight", "bw_diff", "distance", "race_number",
     "n_runners", "month"] + cat_cols +
    ["prev_finish_1", "prev_finish_2", "prev_finish_3", "prev_finish_ratio", "avg_finish_ratio_5",
     "prev_speed", "avg_speed_5", "max_speed_5",
     "top3_rate_5", "win_rate_10", "prev_time_behind", "avg_time_behind_5", "prev_last3f_rank",
     "avg_last3f_rank_5", "prev_corner1_ratio", "avg_corner1_ratio_5", "prev_popularity",
     "prev_log_odds", "n_races", "days_since_last", "dist_diff", "weight_diff", "same_venue_as_prev",
     "jockey_key_top3_rate", "jockey_key_win_rate", "trainer_top3_rate"] +
    [c for c in df.columns if c.endswith("_inrace")] + F4_COLS
)
if USE_ODDS:
    features.append("odds")
print(len(features), "features")

## ④-2 影のモデル用だった特徴量 6個（2026/11 から本番のモデルに入れる）

3年×乱数10回の検証で精度 29勝1敗（2026/11）。最終確認（影＋B2＋市場を出発点にしたレース単位の学習）で30勝0敗だったため、本番のモデルに入れる。
すべて「そのレースより前」の成績から作る（予想ノートブックと同じ計算）。

In [ ]:
import time
t0 = time.time()
APT_TOP = ["field_vs_prev_field", "avg_field_level_3", "apt_course_speed_diff", "jt_top3_rate",
           "prev_speed_vs_avg5", "apt_going_speed_diff"]
APT_BANDS = [0, 1200, 1600, 2000, 9999]
APT_K = 30
_s = df.sort_values(["date", "race_id", "number"])
_band = pd.cut(_s["distance"], APT_BANDS, labels=False).astype("float")
_going = np.where(_s["track_condition"].astype(str).isin(["重", "不良"]), 1, 0)
_sp = _s["speed"].fillna(0).astype(float)
_ok = _s["speed"].notna().astype(float)
def _prior_mean(keys):
    g_s = _sp.groupby(keys, sort=False).cumsum() - _sp
    g_n = _ok.groupby(keys, sort=False).cumsum() - _ok
    return g_s / g_n.replace(0, np.nan)
_all = _prior_mean([_s["horse_key"]])
_course = _prior_mean([_s["horse_key"], _s["venue"].astype(str), _s["course_type"].astype(str), _band.fillna(-1)])
_goingm = _prior_mean([_s["horse_key"], pd.Series(_going, index=_s.index)])
df["apt_course_speed_diff"] = _course - _all
df["apt_going_speed_diff"] = _goingm - _all
_fl = _s.groupby("horse_key", sort=False)["field_level"]
df["prev_field_level_apt"] = _fl.shift(1)
df["avg_field_level_3"] = _fl.transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
df["field_vs_prev_field"] = df["field_level"] - df["prev_field_level_apt"]
df["prev_speed_vs_avg5"] = df["prev_speed"] - df["avg_speed_5"]
_t3 = df["top3"].fillna(0).astype(float)
APT_PRI3 = float(_t3.mean())
_daily = (df.assign(_t3=_t3).groupby(["jockey_key", "trainer", "date"], observed=True)["_t3"]
          .agg(["sum", "count"]).reset_index().sort_values(["jockey_key", "trainer", "date"]))
_gg = _daily.groupby(["jockey_key", "trainer"], observed=True)
_daily["jt_top3_rate"] = ((_gg["sum"].cumsum() - _daily["sum"]) + APT_PRI3 * APT_K) / ((_gg["count"].cumsum() - _daily["count"]) + APT_K)
df["jt_top3_rate"] = df[["jockey_key", "trainer", "date"]].merge(
    _daily[["jockey_key", "trainer", "date", "jt_top3_rate"]], on=["jockey_key", "trainer", "date"], how="left")["jt_top3_rate"].values
print(f"③上位の特徴量 {len(APT_TOP)}個 / {time.time() - t0:.0f}秒")
print(df[APT_TOP].describe().T[["count", "mean", "min", "max"]].round(3).to_string())


## ④-3 B2 ペースと展開（2026/11 から本番のモデルに入れる）

前走のペースの速さ（前半の時計）、上がりの速さ（コースの基準との差）、「速いペースを前で受けた」度合い、今回のメンバーの先行馬の数と自分の位置取り。3年×乱数10回で 26勝4敗。
基準タイムは、データの最終年の2年前より前のレースだけで作る（検証と同じ）。

In [ ]:
t0 = time.time()
LAST = int(df["year"].max())
df = df.sort_values(["date", "race_id", "number"]).reset_index(drop=True)
df["_first"] = df["time_sec"] - df["last_3f"]                 # 前半（上がり3ハロンより前）の時計
base_p = df[(df["year"] < LAST - 2) & (df["finish"] <= 5)].groupby("_bk").agg(f_m=("_first", "mean"), f_s=("_first", "std"),
                                                                              l_m=("last_3f", "mean"), l_s=("last_3f", "std"), n=("_first", "size"))
base_p = base_p[base_p["n"] >= 30]     # 基準は評価する年より前のデータだけで作る
top3_first = df["_first"].where(df["finish"] <= 3).groupby(df["race_id"]).transform("median")
df["race_pace_z"] = (df["_bk"].map(base_p["f_m"]) - top3_first) / df["_bk"].map(base_p["f_s"])     # プラス＝前半が速いレース
df["l3z"] = (df["_bk"].map(base_p["l_m"]) - df["last_3f"]) / df["_bk"].map(base_p["l_s"])          # プラス＝上がりが速い
df["pace_x_pos"] = df["race_pace_z"] * (1 - df["corner1_ratio"])                                  # 速いペースを前で受けたほど大きい
g = df.groupby("horse_key", sort=False)
df["prev_pace_z"] = g["race_pace_z"].shift(1)
df["prev_l3z"] = g["l3z"].shift(1)
df["avg_l3z_5"] = g["l3z"].shift(1).groupby(df["horse_key"]).rolling(5, min_periods=1).mean().reset_index(level=0, drop=True)
df["prev_pace_x_pos"] = g["pace_x_pos"].shift(1)
df["avg_pace_x_pos_5"] = g["pace_x_pos"].shift(1).groupby(df["horse_key"]).rolling(5, min_periods=1).mean().reset_index(level=0, drop=True)
front = (df["avg_corner1_ratio_5"] <= 0.3).astype(float)
df["field_front_n"] = front.groupby(df["race_id"]).transform("sum")
df["front_share"] = df["field_front_n"] / df["n_runners"]
df["c1_rank_inrace"] = df.groupby("race_id")["avg_corner1_ratio_5"].rank(pct=True)
df["front_comp"] = front * (df["field_front_n"] - 1)
B2 = ["prev_pace_z", "prev_l3z", "avg_l3z_5", "prev_pace_x_pos", "avg_pace_x_pos_5", "field_front_n", "front_share", "c1_rank_inrace", "front_comp"]
print(f"B2 {len(B2)}個 / ペースが計算できた割合 {df['race_pace_z'].notna().mean():.1%} / {time.time() - t0:.0f}秒")
# 予想ノートブックで、新しく取り込んだ結果から同じ値を計算するための基準（競馬場・コース・距離・馬場ごと）
_k = df.drop_duplicates("_bk").set_index("_bk")[["venue", "course_type", "distance", "track_condition"]]
BASE_P = {(str(_k.at[b, "venue"]), str(_k.at[b, "course_type"]), int(float(_k.at[b, "distance"])), str(_k.at[b, "track_condition"])):
          (float(r["f_m"]), float(r["f_s"]), float(r["l_m"]), float(r["l_s"])) for b, r in base_p.iterrows() if b in _k.index}

# 本番のモデルの特徴量 = 今までの特徴量 ＋ 影の6個 ＋ B2 の9個。今までの特徴量は旧モデル（影のモデルとして残す）用に取っておく
FEATURES_OLD = list(features)
features = FEATURES_OLD + APT_TOP + B2
print(f"本番のモデルの特徴量: {len(features)}個（今まで {len(FEATURES_OLD)} ＋ 影の6個 ＋ B2 {len(B2)}個）")

## ⑤ 最終モデルの学習

データの最終年を検証用（早期終了と市場との合成の重み）にして、それより前の全データで学習する。
「1着確率」と「3着内確率」の2本。

保存するモデルは ⑤-7 で最終年も含めて学習し直したもの（2026/11 変更）。

**2026/11 から**: 1着のモデルは「市場の確率を出発点にして、1レースの中でどの馬が勝つかを学習する」方式（市場を出発点にしたレース単位の学習）。
予想では、締切前のオッズから出した市場の確率に、モデルが学んだ「市場とのずれ」を足して1着確率を出す。3着内のモデルは今までどおり。

In [ ]:
from scipy.optimize import minimize
import io, re, joblib

df["year"] = df["date"].dt.year
inv = 1 / df["odds"].where(df["odds"] > 0)
df["p_mkt"] = inv / inv.groupby(df["race_id"]).transform("sum")
df["race_ok"] = (df.groupby("race_id")["p_mkt"].transform(lambda s: s.notna().all())
                 & (df.groupby("race_id")["win"].transform("sum") == 1))

def model_proba(m, X):
    # 学習済みモデル（1つ、または乱数違いの複数）の確率の平均（1頭ずつ学習するモデル用: 3着内・影のモデル）
    ms = m if isinstance(m, (list, tuple)) else [m]
    return np.mean([x.predict_proba(X)[:, 1] for x in ms], axis=0)

# ---------- 1着のモデル: 市場を出発点にしたレース単位の学習 ----------
ORDER = ["date", "race_id", "number"]
def _codes(d):
    return pd.factorize(d["race_id"])[0]
def _logp(d):
    return np.log(d["p_mkt"].clip(1e-6).fillna(1e-6).values)
def _soft(codes, s):
    mx = np.full(codes.max() + 1, -np.inf)
    np.maximum.at(mx, codes, s)
    e = np.exp(s - mx[codes])
    return e / np.bincount(codes, weights=e)[codes]
def _soft_obj(codes):
    def f(y, s):
        p = _soft(codes, np.asarray(s, dtype=float))
        return p - np.asarray(y, dtype=float), np.maximum(p * (1 - p), 1e-6)
    return f
def _soft_metric(codes_by_len):
    def f(y, s):
        c = codes_by_len.get(len(y))
        if c is None:
            return "race_ll", 0.0, False
        p = _soft(c, np.asarray(s, dtype=float))
        return "race_ll", float(-np.log(np.clip(p[np.asarray(y) == 1], 1e-9, None)).mean()), False
    return f
def win_proba(ms, d):
    # 1着確率（レース内で合計1）。乱数違いのモデルの平均。d には features・p_mkt・race_id が必要
    c, lp = _codes(d), _logp(d)
    return np.mean([_soft(c, m.predict(d[features]) + lp) for m in ms], axis=0)

def combine(d, w):
    s = w[0] * np.log(d["p_win_n"].clip(1e-6)) + w[1] * np.log(d["p_mkt"].clip(1e-6))
    e = np.exp(s - s.groupby(d["race_id"]).transform("max"))
    return e / e.groupby(d["race_id"]).transform("sum")

def logloss(w, d):
    return -np.log(combine(d, w)[d["win"] == 1].clip(1e-9)).mean()

LAST = int(df["year"].max())
tr = df[df["year"] <= LAST - 1].sort_values(ORDER)
va = df[(df["year"] == LAST) & df["race_ok"]].sort_values(ORDER).copy()
params = dict(n_estimators=3000, learning_rate=0.05, num_leaves=63, min_child_samples=100,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)
SEEDS = [42, 43, 44]   # 乱数を変えて3回学習し、予想時はその平均を使う（学習ごとの揺れを抑える）
final_models = {}
ms, BEST_IT_WIN = [], []
for sd in SEEDS:
    m = lgb.LGBMRegressor(**{**params, "random_state": sd, "objective": _soft_obj(_codes(tr)), "metric": "None"})
    m.fit(tr[features], tr["win"].astype(float), init_score=_logp(tr),
          eval_set=[(va[features], va["win"].astype(float))], eval_init_score=[_logp(va)],
          eval_metric=_soft_metric({len(va): _codes(va)}), callbacks=[lgb.early_stopping(100, verbose=False)])
    ms.append(m.booster_); BEST_IT_WIN.append(m.best_iteration_)
final_models["win"] = ms
print("win（市場を出発点・レース単位）木の数:", " / ".join(map(str, BEST_IT_WIN)), f"（平均 {np.mean(BEST_IT_WIN):.0f}）")
for target in ["top3"]:
    ms = []
    for sd in SEEDS:
        m = lgb.LGBMClassifier(**{**params, "random_state": sd})
        m.fit(tr[features], tr[target], eval_set=[(va[features], va[target])], eval_metric="binary_logloss",
              callbacks=[lgb.early_stopping(100, verbose=False)])
        ms.append(m)
    final_models[target] = ms
    its = [m.best_iteration_ for m in ms]
    print(target, "木の数:", " / ".join(map(str, its)), f"（平均 {np.mean(its):.0f}）")

va["p_win"] = win_proba(final_models["win"], va)
va["p_win_n"] = va["p_win"]
w_final = minimize(logloss, x0=[0.5, 0.5], args=(va,), method="Nelder-Mead").x
print(f"合成の重み: モデル {w_final[0]:.3f} / 市場 {w_final[1]:.3f}")
print(f"{LAST}年の対数損失: 市場 {logloss([0, 1], va):.4f} → 合成 {logloss(w_final, va):.4f}")
print("データの最終日:", df["date"].max().date())

# ワイドの「補正後の確率」用: 2着・3着を決めるときの強さの差の縮め方（λ）を、最終年のレースで決める
# 2着の確率 = p_j^λ2 / Σ(1着馬以外) p_i^λ2、3着 = p_j^λ3 / Σ(1・2着馬以外) p_i^λ3 として、実際の2・3着に最も当てはまる λ
va["p_comb_w"] = combine(va, w_final).values
_races = []
for _, X_ in va.groupby("race_id", sort=False):
    X_ = X_[X_["finish"].notna()].sort_values("finish")
    if len(X_) >= 4 and X_["finish"].iloc[:3].tolist() == [1, 2, 3]:
        _races.append(np.log(np.clip(X_["p_comb_w"].values, 1e-9, None)))
def _nll_lam(lam, pos):
    tot = 0.0
    for lp in _races:
        r_ = lam * lp[pos - 1:]
        tot -= r_[0] - np.log(np.exp(r_ - r_.max()).sum()) - r_.max()
    return tot / len(_races)
WIDE_LAM = (float(minimize(lambda x: _nll_lam(x[0], 2), x0=[1.0], method="Nelder-Mead").x[0]),
            float(minimize(lambda x: _nll_lam(x[0], 3), x0=[1.0], method="Nelder-Mead").x[0]))
print(f"ワイドの補正後の確率: 2着λ={WIDE_LAM[0]:.3f} / 3着λ={WIDE_LAM[1]:.3f}（{len(_races):,}レースから）")

# 予想時に使う名前→IDの対応表（全角・半角スペースや見習い記号を除いて照合）
def norm_name(s):
    return s.astype(str).str.replace(r"[\s\u3000▲△☆◇★]", "", regex=True)

name2horse  = df.assign(n=norm_name(df["horse_name"])).sort_values("date").groupby("n")["horse_key"].last()
name2jockey = df.assign(n=norm_name(df["jockey_name"])).sort_values("date").groupby("n")["jockey_key"].last()
name2trainer = df.assign(n=norm_name(df["trainer"])).sort_values("date").groupby("n")["trainer"].last()
print("使える値: ")
for c in ["venue", "course_type", "turn", "weather", "track_condition"]:
    print(f"  {c}: {list(df[c].cat.categories)}")

## ⑤-2 2026年のレースでの成績（◎の1着率・1番人気との比較・単勝の回収率）

In [ ]:
# ---------- 2026年のレースでの成績（初期版と比べる用） ----------
ev_df = va.copy()
ev_df["p"] = combine(ev_df, w_final)
ev_df["ev"] = ev_df["p"] * ev_df["odds"]
ev_df["finish"] = pd.to_numeric(ev_df["finish"], errors="coerce")
top = ev_df.loc[ev_df.groupby("race_id")["p"].idxmax()]               # ◎（モデル＋市場の1着確率が最も高い馬）
fav = ev_df.loc[ev_df.groupby("race_id")["odds"].idxmin()]            # 1番人気
n = len(top)
print(f"\n【{LAST}年 {n:,}レースでの成績】")
print(f"◎        1着率 {top['win'].mean():.1%} / 3着内率 {top['top3'].mean():.1%} / 単勝を毎回買った回収率 {(top['odds'] * top['win']).mean():.1%}")
print(f"1番人気  1着率 {fav['win'].mean():.1%} / 3着内率 {fav['top3'].mean():.1%} / 単勝を毎回買った回収率 {(fav['odds'] * fav['win']).mean():.1%}")
print(f"◎と1番人気が違う馬だったレース: {(top['horse_key'].values != fav['horse_key'].values).mean():.1%}")
b = ev_df.loc[ev_df.groupby("race_id")["ev"].idxmax()]
b = b[(b["ev"] >= 1.1) & (b["odds"] <= 20)]                           # 自動予想と同じ買い条件
if len(b):
    print(f"単勝の買い（期待値≧1.1・20倍以下）: {len(b):,}回 / 的中 {b['win'].mean():.1%} / 回収率 {(b['odds'] * b['win']).mean():.1%} / 平均オッズ {b['odds'].mean():.1f}倍")
print("※ 2026年は合成の重みを決めるのにも使っているので、実際の運用より少し良く出やすい（初期版との比較には使える）")
# ---------- レース番号（1日の前半・後半）ごと、競馬場ごとの内訳 ----------
def breakdown(key, label):
    rows = []
    for k, t in top.groupby(key, observed=True):
        f = fav[fav[key] == k]
        bb = b[b[key] == k]
        rows.append({label: k, "レース数": len(t), "平均頭数": round(t["n_runners"].mean(), 1),
                     "◎1着率": f"{t['win'].mean():.1%}", "1番人気1着率": f"{f['win'].mean():.1%}",
                     "◎と1番人気が違う": f"{(t['horse_key'].values != f['horse_key'].values).mean():.1%}",
                     "買い回数": len(bb), "買い的中率": f"{bb['win'].mean():.1%}" if len(bb) else "-",
                     "買い回収率": f"{(bb['odds'] * bb['win']).mean():.0%}" if len(bb) else "-"})
    print(pd.DataFrame(rows).to_string(index=False))

for d_ in (top, fav, b):
    d_["R帯"] = pd.cut(d_["race_number"], [0, 4, 8, 99], labels=["1〜4R", "5〜8R", "9R以降"])
print("\n【レース番号ごと】（後ろのレースほどクラスが高く、メイン・ナイターが多い）")
breakdown("R帯", "R帯")
print("\n【競馬場ごと】")
breakdown("venue", "競馬場")
print("※ 買いの回数が100回未満の行は、回収率が偶然で大きく動く（±20%以上）ので参考程度に")


## ⑤-3 期待値 0.05刻みの検証（後半だけで採点）

In [ ]:
# ---------- 期待値 0.05刻みの検証（2026年前半で合成の重みを決め、後半だけで採点）----------
from scipy.optimize import minimize
v3 = va.copy()
days = np.sort(v3["date"].unique())
mid = days[len(days) // 2]
first, second = v3[v3["date"] < mid], v3[v3["date"] >= mid].copy()
w_h = minimize(logloss, x0=[0.5, 0.5], args=(first,), method="Nelder-Mead").x
second["p"] = combine(second, w_h).values
second["ev"] = second["p"] * second["odds"]
EDGES = [0.90, 1.00, 1.05, 1.10, 1.15, 1.20, 1.25, 1.30, 1.35, 1.40, 1.45, 1.50, 1.60, 1.80, 2.00, 99]

def ev_table(d, title):
    d = d.copy()
    d["区間"] = pd.cut(d["ev"], EDGES, right=False)
    rows = []
    for b, s in d.groupby("区間", observed=True):
        up = d[d["ev"] >= b.left]
        rows.append({"期待値": f"{b.left:.2f}〜" + ("" if b.right > 50 else f"{b.right:.2f}"), "回数": len(s),
                     "的中": int(s["win"].sum()), "実際の的中率": f"{s['win'].mean():.1%}", "予想の的中率": f"{s['p'].mean():.1%}",
                     "平均オッズ": round(s["odds"].mean(), 1), "回収率": f"{(s['odds'] * s['win']).mean():.0%}",
                     "この値以上で買うと 回数": len(up), "回収率 ": f"{(up['odds'] * up['win']).mean():.0%}"})
    print(title)
    print(pd.DataFrame(rows).to_string(index=False))

best = second.loc[second.groupby("race_id")["ev"].idxmax()]
best = best[best["odds"] <= 20]
print(f"\n【期待値ごとの成績】{LAST}年後半 {second['race_id'].nunique():,}レース（{pd.Timestamp(mid).date()}〜）"
      f"／合成の重みは前半で決定（モデル {w_h[0]:.3f} / 市場 {w_h[1]:.3f}）")
ev_table(best, "\n▼ 自動予想と同じ買い方（各レースで期待値が最大の1頭・20倍以下）")
ev_table(second[second["odds"] <= 20], "\n▼ 参考: 20倍以下の全出走馬（データが多いぶん傾向が見やすい）")
print("※ 判断も払戻も確定オッズで計算している。本番は5分前オッズで判断するので、締切までのオッズの下落ぶん成績は下がる")
print("※ 1区間が100回未満だと回収率は偶然で ±20% 以上動く。隣どうしの区間の差より、全体の傾きを見る")


## ⑤-5 単勝: 1着率×期待値ごとの回収率（新しい買いの基準づくり用）

In [ ]:
# ---------- 単勝: 1着率 × 期待値 の組み合わせごとの回収率 ----------
# 対象は全出走馬（1レースで複数頭が当てはまることもある）。1頭100円ずつ買ったとして計算。
# 新しい買いの基準を作るときは、「前半」と「後半」の両方で100%を超えるマスだけを候補にする
# （片方だけ良いマスは偶然の可能性が高い。前半は合成の重みを決めたデータなので少し甘め、後半が本番に近い）
first_g = first.copy()
first_g["p"] = combine(first_g, w_h).values
first_g["ev"] = first_g["p"] * first_g["odds"]
P_EDGES = [0, .05, .10, .15, .20, .25, .30, .35, .40, .45, .50, 1.01]
P_LAB = ["0-5%", "5-10%", "10-15%", "15-20%", "20-25%", "25-30%", "30-35%", "35-40%", "40-45%", "45-50%", "50%-"]
E_EDGES = [0, .50, .75, 1.00, 1.25, 1.50, 2.00, 99]
E_LAB = ["〜0.5", "0.5-0.75", "0.75-1.0", "1.0-1.25", "1.25-1.5", "1.5-2.0", "2.0〜"]

def grid(d):
    d = d.assign(P帯=pd.cut(d["p"], P_EDGES, labels=P_LAB, right=False),
                 期待値帯=pd.cut(d["ev"], E_EDGES, labels=E_LAB, right=False),
                 ret=d["odds"] * d["win"])
    g = d.groupby(["P帯", "期待値帯"], observed=False)
    n = g.size().unstack()
    r = (g["ret"].mean().unstack() * 100).round(0)
    h = (g["win"].mean().unstack() * 100).round(1)
    return n, r, h

def show(df_, title):
    print(title)
    print(df_.to_string(na_rep="-"))

n1, r1, h1 = grid(first_g)
n2, r2, h2 = grid(second)
n_all, r_all, h_all = grid(pd.concat([first_g, second]))
print(f"\n【単勝 1着率×期待値】前半 {first_g['race_id'].nunique():,}レース / 後半 {second['race_id'].nunique():,}レース（後半は {pd.Timestamp(mid).date()}〜）")
show(n_all, "\n▼ 頭数（前半＋後半）")
show(h_all, "\n▼ 実際の1着率 %（前半＋後半）")
show(r_all, "\n▼ 回収率 %（前半＋後半）")
show(r1, "\n▼ 回収率 %（前半だけ）")
show(r2, "\n▼ 回収率 %（後半だけ）")
ok = (n1 >= 50) & (n2 >= 50) & (r1 > 100) & (r2 > 100)
cells = [(p_, e_, int(n_all.loc[p_, e_]), r1.loc[p_, e_], r2.loc[p_, e_]) for p_ in P_LAB for e_ in E_LAB if bool(ok.loc[p_, e_])]
print("\n▼ 前半・後半とも50頭以上かつ回収率100%超のマス（新しい基準の候補）")
print("\n".join(f"  1着率 {p_} × 期待値 {e_}: {n_}頭 / 前半 {a:.0f}% / 後半 {b:.0f}%" for p_, e_, n_, a, b in cells) or "  なし")
print("※ 判断も払戻も確定オッズで計算（本番は5分前・3分前オッズで判断するので、締切までの下落ぶん下がる）")
print("※ 1マス数百頭でも回収率は ±15〜30% 動く。マスを選びすぎると偶然に合わせた基準になるので、隣り合うマスがまとまって良い所を探す")


## ⑤-6 単勝の買い基準の候補を比べる（前半・後半それぞれで採点）

In [ ]:
# ---------- 単勝の買い基準の候補を比べる（前半・後半それぞれで） ----------
# 各案について、2026年前半（合成の重みを決めたデータ＝少し甘め）と後半（本番に近い）で別々に採点する。
# 「ぶれ幅」はレース単位で1000回リサンプリングした回収率の95%範囲。前半・後半とも100%を超え、ぶれ幅の下限も高い案ほど信頼できる。
def pick(d, ev_min, p_min=0.0, p_max=1.01, odds_max=999, one_per_race=False):
    s = d[(d["ev"] >= ev_min) & (d["p"] >= p_min) & (d["p"] < p_max) & (d["odds"] <= odds_max)]
    if one_per_race and len(s):
        s = s.loc[s.groupby("race_id")["ev"].idxmax()]
    return s

RULES = [
    ("今の基準: 期待値1.1以上・20倍以下・1レース1頭", dict(ev_min=1.1, odds_max=20, one_per_race=True)),
    ("案A: 期待値1.0以上・1着率10%以上", dict(ev_min=1.0, p_min=0.10)),
    ("案B: 案A＋1レース1頭（期待値最大）", dict(ev_min=1.0, p_min=0.10, one_per_race=True)),
    ("案C: 期待値1.1以上・1着率10%以上", dict(ev_min=1.1, p_min=0.10)),
    ("案D: 期待値1.0以上・1着率15%以上", dict(ev_min=1.0, p_min=0.15)),
    ("案E: 期待値1.0以上・1着率10〜40%", dict(ev_min=1.0, p_min=0.10, p_max=0.40)),
    ("参考: 期待値1.0以上・1着率10%未満", dict(ev_min=1.0, p_max=0.10)),
]
rng_b = np.random.default_rng(0)

def roi_ci(s, B=1000):
    if len(s) == 0:
        return np.nan, np.nan
    per = (s["odds"] * s["win"]).groupby(s["race_id"]).agg(["sum", "size"])
    v, n = per["sum"].values, per["size"].values
    idx = rng_b.integers(0, len(per), size=(B, len(per)))
    r = v[idx].sum(1) / n[idx].sum(1)
    return np.percentile(r, 2.5), np.percentile(r, 97.5)

rows = []
for name, kw in RULES:
    row = {"案": name}
    for lab, d in [("前半", first_g), ("後半", second)]:
        s = pick(d, **kw)
        lo, hi = roi_ci(s)
        row[f"{lab} 頭数"] = len(s)
        row[f"{lab} 1日あたり"] = round(len(s) / max(d["date"].nunique(), 1), 1)
        row[f"{lab} 的中率"] = f"{s['win'].mean():.1%}" if len(s) else "-"
        row[f"{lab} 回収率"] = f"{(s['odds'] * s['win']).mean():.0%}" if len(s) else "-"
        row[f"{lab} ぶれ幅"] = f"{lo:.0%}〜{hi:.0%}" if len(s) else "-"
    s_all = pick(pd.concat([first_g, second]), **kw)
    r_all = (s_all["odds"] * s_all["win"]).mean() if len(s_all) else np.nan
    row["全体 回収率"] = f"{r_all:.0%}" if len(s_all) else "-"
    row["平均オッズ"] = round(s_all["odds"].mean(), 1) if len(s_all) else "-"
    row["払戻が何%減ると100%割れ"] = f"{(1 - 1 / r_all):.0%}" if len(s_all) and r_all > 1 else "既に100%未満"
    rows.append(row)
RT = pd.DataFrame(rows)
print(f"\n【単勝の買い基準の比較】前半 {first_g['race_id'].nunique():,}レース（〜{pd.Timestamp(mid).date()}）/ 後半 {second['race_id'].nunique():,}レース")
cols_h = lambda lab: [f"{lab} 頭数", f"{lab} 1日あたり", f"{lab} 的中率", f"{lab} 回収率", f"{lab} ぶれ幅"]
print("\n▼ 後半（本番に近い）")
print(RT[["案"] + cols_h("後半")].to_string(index=False))
print("\n▼ 前半（合成の重みを決めたデータ・少し甘め）")
print(RT[["案"] + cols_h("前半")].to_string(index=False))
print("\n▼ 全体と余裕")
print(RT[["案", "全体 回収率", "平均オッズ", "払戻が何%減ると100%割れ"]].to_string(index=False))

# 月ごとの安定性（全体で回収率が最も高い案と、今の基準）
def monthly(kw):
    s = pick(pd.concat([first_g, second]), **kw).assign(月=lambda x: x["date"].dt.strftime("%Y-%m"))
    return s.groupby("月").apply(lambda g: f"{(g['odds'] * g['win']).mean():.0%}（{len(g)}）")
cand = [k for k in range(1, len(RULES) - 1)]
best_k = max(cand, key=lambda k: float(RT.loc[k, "全体 回収率"].rstrip("%")) if RT.loc[k, "全体 回収率"] != "-" else -1)
print(f"\n▼ 月ごとの回収率（頭数）: {RULES[0][0]} と {RULES[best_k][0]}")
print(pd.DataFrame({"今の基準": monthly(RULES[0][1]), RULES[best_k][0][:3]: monthly(RULES[best_k][1])}).to_string())
print("\n※ 判断も払戻も確定オッズで計算。本番の締切直前のオッズの下落（運用2週間では勝ち馬で約4割）は含まない")
print("※「払戻が何%減ると100%割れ」= この案の余裕。本番のオッズの下落がこれより大きければ、本番では負ける")


## ⑤-4 ワイドの度合い 0.05刻みの検証（後半だけで採点）

In [ ]:
# ---------- ワイドの検証（⑤-3 と同じく、2026年前半で合成の重みを決め、後半だけで採点）----------
# 自動予想と同じ計算: 合成後の1着確率から着順をシミュレーションし、2頭がそろって複勝圏に入る確率（ワイド確率）を出す。
# 「度合い」= モデルのワイド確率 ÷ 単勝オッズから見た市場のワイド確率。今の買い条件は 度合い≧1.5 かつ ワイド確率≧0.2
import re
N_SIM_V = 2000
wp = pays[pays["bet_type"] == "ワイド"].copy()
def _pk(s):
    nums = sorted(int(x) for x in re.findall(r"\d+", str(s)))
    return f"{nums[0]}-{nums[1]}" if len(nums) == 2 else None
wp["key"] = wp["horse_num"].map(_pk)
wp["payout"] = pd.to_numeric(wp["payout"].astype(str).str.replace(",", ""), errors="coerce")
WPAY = wp.dropna(subset=["key", "payout"]).groupby(["race_id", "key"])["payout"].max().to_dict()
have_wide = set(wp["race_id"])

rng = np.random.default_rng(0)
recs = []
for rid, X in second.groupby("race_id", sort=False):
    n = len(X)
    if n < 5 or rid not in have_wide:
        continue
    k = 3 if n >= 8 else 2
    gum = rng.gumbel(size=(N_SIM_V, n))
    mats = {}
    for name, col in (("m", "p"), ("k", "p_mkt")):
        g = gum + np.log(np.clip(X[col].values, 1e-9, None))
        top = np.argpartition(-g, k - 1, axis=1)[:, :k]
        hit = np.zeros((N_SIM_V, n), dtype=np.float32)
        np.put_along_axis(hit, top, 1.0, axis=1)
        mats[name] = (hit.T @ hit) / N_SIM_V
    iu = np.triu_indices(n, 1)
    nums = X["number"].astype(int).values
    a, b = np.minimum(nums[iu[0]], nums[iu[1]]), np.maximum(nums[iu[0]], nums[iu[1]])
    for x, y, pm_, pk_ in zip(a, b, mats["m"][iu], mats["k"][iu]):
        pay_ = WPAY.get((rid, f"{x}-{y}"), 0)
        recs.append((rid, pm_, pk_, pay_))
WR = pd.DataFrame(recs, columns=["race_id", "pw", "pw_mkt", "pay"])
WR["ratio"] = WR["pw"] / WR["pw_mkt"].clip(1e-6)
WR["hit"] = (WR["pay"] > 0).astype(int)

def wide_table(d, edges, key, fmt, title):
    d = d.copy()
    d["区間"] = pd.cut(d[key], edges, right=False)
    rows = []
    for b, s in d.groupby("区間", observed=True):
        up = d[d[key] >= b.left]
        rows.append({"区間": fmt(b), "点数": len(s), "的中": int(s["hit"].sum()), "実際の的中率": f"{s['hit'].mean():.1%}",
                     "予想の的中率": f"{s['pw'].mean():.1%}", "的中時の平均払戻": int(s.loc[s['hit'] == 1, 'pay'].mean()) if s['hit'].any() else 0,
                     "回収率": f"{s['pay'].mean() / 100:.0%}", "この値以上で買うと 点数": len(up), "回収率 ": f"{up['pay'].mean() / 100:.0%}"})
    print(title)
    print(pd.DataFrame(rows).to_string(index=False))

R_EDGES = [1.20, 1.25, 1.30, 1.35, 1.40, 1.45, 1.50, 1.55, 1.60, 1.65, 1.70, 1.75, 1.80, 1.85, 1.90, 1.95, 2.00, 2.25, 2.50, 99]
rfmt = lambda b: f"{b.left:.2f}〜" + ("" if b.right > 50 else f"{b.right:.2f}")
cand = WR[WR["pw"] >= 0.2]
best_w = cand.loc[cand.groupby("race_id")["ratio"].idxmax()]
print(f"\n【ワイドの度合いごとの成績】{LAST}年後半 {WR['race_id'].nunique():,}レース（払戻データのあるレース）")
wide_table(best_w, R_EDGES, "ratio", rfmt, "\n▼ 自動予想の1点目と同じ選び方（ワイド確率≧0.2 の中で度合いが最大の組・各レース1点）")
wide_table(cand, R_EDGES, "ratio", rfmt, "\n▼ 参考: ワイド確率≧0.2 の全部の組")
wide_table(WR[WR["ratio"] >= 1.5], [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 1.01], "pw",
           lambda b: f"{b.left:.2f}〜{b.right:.2f}", "\n▼ 参考: 度合い≧1.5 の組を、ワイド確率ごとに（確率の下限 0.2 が妥当かを見る）")
print("※ 払戻は確定。市場のワイド確率は確定の単勝オッズから計算（本番は5分前の単勝オッズ）")
print("※ 1区間が100点未満だと回収率は偶然で大きく動く。全体の傾きと「この値以上で買うと」を見る")


## ⑤-7 最終年も含めた全データで学習し直す（保存するのはこちら）

⑤〜⑤-4 の成績は、最終年を学習に使っていないモデルでの「正直な」数字。
保存するモデルは、⑤で決めた木の数（＋5%）のまま、**最終年も含めた全データ**で学習し直す（最新のレースも学習の例に入る）。
合成の重みと補正の強さは ⑤ で決めたものをそのまま使う。`FULL_REFIT = False` にすると、今までどおり最終年を除いたモデルを保存する。

In [ ]:
FULL_REFIT = True

if FULL_REFIT:
    import time
    t0 = time.time()
    full = df.sort_values(ORDER)
    refit = {}
    ms, nts = [], []
    for it, sd in zip(BEST_IT_WIN, SEEDS):
        n_tree = max(50, int(round(it * 1.05)))
        m = lgb.LGBMRegressor(**{**params, "random_state": sd, "n_estimators": n_tree, "objective": _soft_obj(_codes(full)), "metric": "None"})
        m.fit(full[features], full["win"].astype(float), init_score=_logp(full))
        ms.append(m.booster_); nts.append(n_tree)
    refit["win"] = ms
    print("win 全データで学習し直し: 木の数", " / ".join(map(str, nts)))
    for target in ["top3"]:
        ms, nts = [], []
        for m0, sd in zip(final_models[target], SEEDS):
            n_tree = max(50, int(round(m0.best_iteration_ * 1.05)))
            m = lgb.LGBMClassifier(**{**params, "random_state": sd, "n_estimators": n_tree})
            m.fit(full[features], full[target])
            ms.append(m); nts.append(n_tree)
        refit[target] = ms
        print(target, "全データで学習し直し: 木の数", " / ".join(map(str, nts)))
    # 念のための確認: 新旧モデルの予想がどれくらい似ているか（最終年のレースで。学習に使ったので成績の比較ではない）
    p_old = win_proba(final_models["win"], va)
    p_new = win_proba(refit["win"], va)
    print(f"新旧モデルの1着確率の相関（{LAST}年）: {np.corrcoef(p_old, p_new)[0, 1]:.3f}（0.95以上なら大きな変化なし）")
    VALID_MODELS = final_models
    final_models = refit
    print(f"学習データ: {full['race_id'].nunique():,}レース（〜{full['date'].max().date()}）/ {time.time() - t0:.0f}秒")
else:
    print(f"最終年（{LAST}年）を除いたモデルをそのまま保存します")

## ⑤-8 影のモデル（旧本番のモデル）を作る

2026/11 から、影のモデルは**これまでの本番のモデル**（今までの特徴量・1頭ずつ学習）。新しい本番のモデルが本番でも上回るかを、同じレース・同じオッズで並べて確かめる。
予想ノートブックが本番と並べて計算し、記録と #新モデル への通知に使う。`APT_SHADOW = False` にすると作らない。

In [ ]:
APT_SHADOW = True
final_models_apt, w_final_apt, features_apt = None, None, None
if APT_SHADOW:
    t0 = time.time()
    features_apt = FEATURES_OLD             # 旧本番のモデル（今までの特徴量）
    tr_a = df[df["year"] <= LAST - 1]
    va_a = df[(df["year"] == LAST) & df["race_ok"]].copy()
    ms_a, its_a = [], []
    for sd in SEEDS:
        m = lgb.LGBMClassifier(**{**params, "random_state": sd})
        m.fit(tr_a[features_apt], tr_a["win"], eval_set=[(va_a[features_apt], va_a["win"])], eval_metric="binary_logloss",
              callbacks=[lgb.early_stopping(100, verbose=False)])
        ms_a.append(m); its_a.append(m.best_iteration_)
    va_a["p_win"] = model_proba(ms_a, va_a[features_apt])
    va_a["p_win_n"] = va_a["p_win"] / va_a.groupby("race_id")["p_win"].transform("sum")
    w_final_apt = minimize(logloss, x0=[0.5, 0.5], args=(va_a,), method="Nelder-Mead").x
    va_m = va_a.copy()
    va_m = va_m.sort_values(ORDER)
    va_m["p_win"] = win_proba(VALID_MODELS["win"] if FULL_REFIT else final_models["win"], va_m)
    va_m["p_win_n"] = va_m["p_win"]
    print("影のモデル（旧本番）木の数:", " / ".join(map(str, its_a)), f"/ 合成の重み モデル {w_final_apt[0]:.3f}・市場 {w_final_apt[1]:.3f}")
    print(f"{LAST}年の対数損失（参考・学習の外）: 市場 {logloss([0, 1], va_a):.4f} / 新しい本番 {logloss(w_final, va_m):.4f} / 影（旧本番） {logloss(w_final_apt, va_a):.4f}")
    if FULL_REFIT:
        full = df.sort_values(["date", "race_id", "number"])
        refit_a = []
        for it, sd in zip(its_a, SEEDS):
            m = lgb.LGBMClassifier(**{**params, "random_state": sd, "n_estimators": max(50, int(round(it * 1.05)))})
            m.fit(full[features_apt], full["win"])
            refit_a.append(m)
        ms_a = refit_a
    final_models_apt = {"win": ms_a}
    print(f"影のモデル: 完了 / {time.time() - t0:.0f}秒")

## ⑥ 予想に必要なものを保存

In [ ]:
HIST_COLS = ["horse_key", "date", "race_id", "number", "finish", "finish_ratio", "speed", "top3", "win",
             "time_behind", "last3f_rank", "corner1_ratio", "popularity", "odds", "distance", "weight", "venue",
             "jockey_key", "trainer", "horse_name",
             "course_type", "track_condition", "field_level",     # 適性・相手関係の特徴量の計算用
             "race_pace_z", "l3z", "pace_x_pos"]                  # B2 ペースと展開の計算用
HIST_DB = df[HIST_COLS].copy()
HIST_DB["venue"] = HIST_DB["venue"].astype(str)
for _c in ["course_type", "track_condition"]:
    HIST_DB[_c] = HIST_DB[_c].astype(str)
CATS = {c: list(df[c].cat.categories) for c in cat_cols}
PRIOR = {t: float(df[t].mean()) for t in ["win", "top3"]}
VENUES = sorted(df["venue"].cat.categories.astype(str), key=len, reverse=True)
DEFAULT_TURN = (df.dropna(subset=["turn"]).groupby("venue", observed=True)["turn"]
                .agg(lambda s: s.astype(str).mode().iloc[0] if len(s) else "右").to_dict())
WEATHERS = list(df["weather"].cat.categories.astype(str))

BUNDLE = "/kaggle/working/keiba_bundle"
os.makedirs(BUNDLE, exist_ok=True)
HIST_DB.to_pickle(f"{BUNDLE}/hist.pkl")
meta = dict(final_models=final_models, w_final=w_final, features=features, cat_cols=cat_cols, CATS=CATS,
            PRIOR=PRIOR, K=K, base=base, BASE_KEYS=BASE_KEYS, name2horse=name2horse, name2jockey=name2jockey,
            name2trainer=name2trainer, VENUES=VENUES, DEFAULT_TURN=DEFAULT_TURN, WEATHERS=WEATHERS,
            WIDE_LAM=WIDE_LAM, full_refit=FULL_REFIT,
            main_kind="soft_init", FEATURES_OLD=FEATURES_OLD, B2=B2, BASE_P=BASE_P, SHADOW_NAME="旧モデル",
            final_models_apt=final_models_apt, w_final_apt=w_final_apt, features_apt=features_apt,
            APT_TOP=APT_TOP, APT_PRI3=APT_PRI3, APT_BANDS=APT_BANDS, APT_K=APT_K, data_last=str(df["date"].max().date()),
            built_at=pd.Timestamp.now(tz="Asia/Tokyo").strftime("%Y-%m-%d %H:%M"))
joblib.dump(meta, f"{BUNDLE}/meta.joblib")
for f in os.listdir(BUNDLE):
    print(f"{f}: {os.path.getsize(f'{BUNDLE}/{f}') / 1e6:.1f} MB")
print("保存しました（データの最終日:", meta["data_last"], "）")